In [8]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS


# ==============================
# Config
# ==============================

@dataclass
class NBEATSPipelineConfig:
    unique_id: str = "house3bed"
    freq: str = "QE"
    h: int = 2
    input_size: int = 8
    max_steps: int = 500
    random_seed: int = 42

    n_windows: int = 4
    step_size: int = 1

    model_dir: str = "models"
    model_name: str = "nbeats_house3bed"

    @property
    def artifact_dir(self) -> str:
        return os.path.join(self.model_dir, self.model_name)

    @property
    def config_path(self) -> str:
        return os.path.join(self.artifact_dir, "config.json")

    @property
    def metrics_path(self) -> str:
        return os.path.join(self.artifact_dir, "metrics.json")


# ==============================
# Monitoring artifacts
# ==============================

@dataclass
class BacktestMetrics:
    mae: float
    rmse: float
    mape: float


@dataclass
class MonitoringSnapshot:
    timestamp: str
    backtest: BacktestMetrics
    window_mae: Optional[float] = None
    window_rmse: Optional[float] = None
    window_mape: Optional[float] = None
    residual_mean: Optional[float] = None
    residual_std: Optional[float] = None
    residual_outlier_share: Optional[float] = None


# ==============================
# Core pipeline
# ==============================

class NBEATSPipeline:
    def __init__(self, config: Optional[NBEATSPipelineConfig] = None):
        self.config = config or NBEATSPipelineConfig()
        os.makedirs(self.config.artifact_dir, exist_ok=True)
        self.nf: Optional[NeuralForecast] = None
        self._last_backtest_metrics: Optional[BacktestMetrics] = None

    def _prepare_df(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "ds" not in out.columns or "y" not in out.columns:
            raise ValueError("DataFrame должен содержать колонки 'ds' и 'y'")

        out = out[["ds", "y"]].copy()
        out["ds"] = pd.to_datetime(out["ds"])
        out = out.sort_values("ds")
        out["unique_id"] = self.config.unique_id
        return out[["unique_id", "ds", "y"]]

    def _build_model(self) -> NeuralForecast:
        model = NBEATS(
            h=self.config.h,
            input_size=self.config.input_size,
            max_steps=self.config.max_steps,
            random_seed=self.config.random_seed,
            alias="NBEATS",
        )
        return NeuralForecast(models=[model], freq=self.config.freq)

    def train(self, df: pd.DataFrame) -> BacktestMetrics:
        ts_df = self._prepare_df(df)
        self.nf = self._build_model()

        cv_df = self.nf.cross_validation(
            df=ts_df,
            n_windows=self.config.n_windows,
            h=self.config.h,
            step_size=self.config.step_size,
        )

        y_true = cv_df["y"].values
        y_pred = cv_df["NBEATS"].values

        mae = float(mean_absolute_error(y_true, y_pred))
        rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)

        nonzero_mask = y_true != 0
        if nonzero_mask.any():
            mape = float(np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])))
        else:
            mape = np.nan

        metrics = BacktestMetrics(mae=mae, rmse=rmse, mape=mape)
        self._last_backtest_metrics = metrics

        self.nf.fit(ts_df)

        self._save_config()
        self._append_metrics(metrics)

        return metrics

    def forecast(self, df: pd.DataFrame, h: Optional[int] = None) -> pd.DataFrame:
        if self.nf is None:
            raise RuntimeError("Модель не загружена и не обучена. Вызови train() или load().")

        ts_df = self._prepare_df(df)
        fcst = self.nf.predict(df=ts_df, h=h)

        out = fcst[fcst["unique_id"] == self.config.unique_id].copy()
        out = out[["ds", "NBEATS"]].rename(columns={"NBEATS": "yhat"})
        return out

    def _save_config(self):
        os.makedirs(self.config.artifact_dir, exist_ok=True)
        with open(self.config.config_path, "w", encoding="utf-8") as f:
            json.dump(asdict(self.config), f, ensure_ascii=False, indent=2)

    def _append_metrics(self, metrics: BacktestMetrics):
        history: List[Dict[str, Any]] = []
        if os.path.exists(self.config.metrics_path):
            with open(self.config.metrics_path, "r", encoding="utf-8") as f:
                history = json.load(f)
        history.append(asdict(metrics))
        with open(self.config.metrics_path, "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)

    def save(self):
        if self.nf is None:
            raise RuntimeError("Нечего сохранять: модель не обучена.")
        self.nf.save(path=self.config.artifact_dir, overwrite=True, save_dataset=True)

    def load(self):
        if os.path.exists(self.config.config_path):
            with open(self.config.config_path, "r", encoding="utf-8") as f:
                cfg_dict = json.load(f)
            self.config = NBEATSPipelineConfig(**cfg_dict)

        os.makedirs(self.config.artifact_dir, exist_ok=True)
        self.nf = NeuralForecast.load(path=self.config.artifact_dir)

    def monitoring_on_window(
        self,
        df_true: pd.DataFrame,
        df_pred: pd.DataFrame,
        outlier_sigma: float = 3.0,
    ) -> MonitoringSnapshot:
        true_df = df_true.copy()
        pred_df = df_pred.copy()

        true_df["ds"] = pd.to_datetime(true_df["ds"])
        pred_df["ds"] = pd.to_datetime(pred_df["ds"])

        merged = pd.merge(true_df, pred_df, on="ds", how="inner")
        if merged.empty:
            raise ValueError("Нет пересечения по датам ds между df_true и df_pred")

        y_true = merged["y"].values
        y_pred = merged["yhat"].values
        residuals = y_true - y_pred

        window_mae = float(mean_absolute_error(y_true, y_pred))
        window_rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)

        nonzero_mask = y_true != 0
        if nonzero_mask.any():
            window_mape = float(np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])))
        else:
            window_mape = np.nan

        resid_mean = float(np.mean(residuals))
        resid_std = float(np.std(residuals)) if len(residuals) > 1 else 0.0

        if resid_std > 0:
            outlier_mask = np.abs(residuals - resid_mean) > outlier_sigma * resid_std
            outlier_share = float(np.mean(outlier_mask))
        else:
            outlier_share = 0.0

        backtest_metrics = self._last_backtest_metrics or BacktestMetrics(
            mae=np.nan, rmse=np.nan, mape=np.nan
        )

        return MonitoringSnapshot(
            timestamp=pd.Timestamp.utcnow().isoformat(),
            backtest=backtest_metrics,
            window_mae=window_mae,
            window_rmse=window_rmse,
            window_mape=window_mape,
            residual_mean=resid_mean,
            residual_std=resid_std,
            residual_outlier_share=outlier_share,
        )


# ==============================
# Data prep helpers
# ==============================

def make_series_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["saledate"] = pd.to_datetime(df["saledate"], dayfirst=True)
    df = df.sort_values("saledate").set_index("saledate")

    ts = df[(df["type"] == "house") & (df["bedrooms"] == 3)][["MA"]].copy()

    out = (
        ts.rename(columns={"MA": "y"})
          .reset_index()
          .rename(columns={"saledate": "ds"})
    )[["ds", "y"]]

    return out


# ==============================
# Example run
# ==============================

if __name__ == "__main__":
    csv_path = "house_property_timeseries.csv"  # положи csv рядом со скриптом

    full_history_df = make_series_df(csv_path)

    train_df = full_history_df.iloc[:-2].copy()
    test_df = full_history_df.iloc[-2:].copy()

    pipe = NBEATSPipeline()

    metrics = pipe.train(train_df)
    print("Backtest metrics:", metrics)

    pipe.save()

    pipe2 = NBEATSPipeline()
    pipe2.load()
    pipe2._last_backtest_metrics = metrics

    fcst = pipe2.forecast(train_df)
    print("\nForecast:")
    print(fcst)

    fcst_window = fcst.merge(test_df[["ds"]], on="ds", how="inner")
    snapshot = pipe2.monitoring_on_window(df_true=test_df, df_pred=fcst_window)
    print("\nMonitoring snapshot:")
    print(snapshot)

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
50        Non-trainable params
2.4 M     Total params
9.552     Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |                                               | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_steps=500` reached.
Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Backtest metrics: BacktestMetrics(mae=5137.328125, rmse=6293.420691484083, mape=0.008236366231031042)


/opt/anaconda3/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |                                             | 0/? [00:00<?, ?it/s]


Forecast:
          ds         yhat
0 2019-06-30  628135.7500
1 2019-09-30  630378.1875

Monitoring snapshot:
MonitoringSnapshot(timestamp='2026-05-30T16:14:43.805750+00:00', backtest=BacktestMetrics(mae=5137.328125, rmse=6293.420691484083, mape=0.008236366231031042), window_mae=2087.53125, window_rmse=2169.5012099558735, window_mape=0.0033072737430236976, residual_mean=2087.53125, residual_std=590.71875, residual_outlier_share=0.0)


In [9]:
import pandas as pd

df = pd.read_csv('house_property_timeseries.csv')
df['saledate'] = pd.to_datetime(df['saledate'], dayfirst=True)
df = df.sort_values('saledate').set_index('saledate')

ts = df[(df['type'] == 'house') & (df['bedrooms'] == 3)][['MA']].copy()

train_df = (
    ts.rename(columns={'MA': 'y'})
      .reset_index()
      .rename(columns={'saledate': 'ds'})
)[['ds', 'y']]

# 1. Инициализация
pipe = NBEATSPipeline()

# 2. Обучение + backtest
metrics = pipe.train(train_df)   # train_df: колонки ds, y
print(metrics)

# 3. Сохранение
pipe.save()

# 4. Загрузка (в другом процессе/скрипте)
pipe2 = NBEATSPipeline()
pipe2.load()

# 5. Прогноз на h шагов
fcst = pipe2.forecast(full_history_df)  # ds, y
print(fcst)

# 6. Мониторинг (когда появились факты за окно прогноза)
snapshot = pipe2.monitoring_on_window(df_true=test_df, df_pred=fcst_window)
print(snapshot)

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
50        Non-trainable params
2.4 M     Total params
9.552     Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |                                               | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_steps=500` reached.
Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


BacktestMetrics(mae=1091.3046875, rmse=1400.2314987172656, mape=0.001737301761918376)


/opt/anaconda3/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |                                             | 0/? [00:00<?, ?it/s]

          ds         yhat
0 2019-12-31  632511.3125
1 2020-03-31  635618.9375
MonitoringSnapshot(timestamp='2026-05-30T16:15:00.555053+00:00', backtest=BacktestMetrics(mae=nan, rmse=nan, mape=nan), window_mae=2087.53125, window_rmse=2169.5012099558735, window_mape=0.0033072737430236976, residual_mean=2087.53125, residual_std=590.71875, residual_outlier_share=0.0)


In [10]:
pipe = NBEATSPipeline()
metrics = pipe.train(train_df)
pipe.save()

pipe2 = NBEATSPipeline()
pipe2.load()
pipe2._last_backtest_metrics = metrics  # или прочитать из файла

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
50        Non-trainable params
2.4 M     Total params
9.552     Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |                                               | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

...

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

Validation: |                                             | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.
Seed set to 42
